# build_silver_epss
`stg_epss_valid` (또는 BACKFILL 모드에서는 Bronze EPSS 과거 파일 전체) × `active_cve_set` → `silver_epss` (grain: `cve_id` × `date`).

- **DAILY**: 오늘 검증된 EPSS 중 `active_cve_set`에 속한 CVE만 필터링해 오늘 파티션만 만든다.
- **BACKFILL**: `target_cve_ids`(신규 active CVE)에 대해 Bronze EPSS 과거 전체를 스캔해 그 CVE들의 이력만 복원한다 — EPSS Bronze를 필터링 없이 전량 보존해온 이유가 바로 이 소급 복원이다(02_Bronze_Silver_저장구조.md §3.1).
- 같은 (`cve_id`,`snapshot_date`)가 재실행으로 다시 들어와도 MERGE로 멱등 처리한다(시나리오 10).

In [ ]:
%run 00_silver_common

In [ ]:
active_cves = spark.table(TBL["active_cve_set"]).select("cve_id")

if run_mode == "BACKFILL":
    if not target_cve_ids:
        raise ValueError("BACKFILL 모드는 target_cve_ids가 필요합니다.")

    hist_df = (
        spark.read.option("header", "true").option("comment", "#")
        .csv(f"{BRONZE_EPSS_PATH}/*/*.csv.gz")
        .withColumn("_file", F.input_file_name())
    )
    cve_col = coalesce_existing(hist_df, ["cve", "cve_id"])
    hist_df = (
        hist_df
        .withColumn("_cve_id", cve_col)
        .withColumn("_epss_score", coalesce_existing(hist_df, ["epss"]).cast(DoubleType()))
        .withColumn("_epss_percentile", coalesce_existing(hist_df, ["percentile"]).cast(DoubleType()))
        .withColumn("_snapshot_date", F.regexp_extract(F.col("_file"), r"(\d{4}-\d{2}-\d{2})", 1))
    )
    scoped = hist_df.filter(
        F.col("_cve_id").isin(target_cve_ids)
        & F.col("_epss_score").isNotNull()
        & F.col("_snapshot_date").isNotNull()
        & (F.col("_snapshot_date") != "")
    )
    source_df = scoped.select(
        F.col("_cve_id").alias("cve_id"),
        F.col("_snapshot_date").alias("snapshot_date"),
        F.col("_epss_score").alias("epss_score"),
        F.col("_epss_percentile").alias("epss_percentile"),
        F.lit(None).cast(StringType()).alias("model_version"),
    )
else:
    source_df = spark.table(TBL["stg_epss_valid"]).select(
        "cve_id", "snapshot_date", "epss_score", "epss_percentile", "model_version"
    )


In [ ]:
filtered = source_df.join(active_cves, on="cve_id", how="inner")

dq_flags = flags_array(
    F.when((F.col("epss_score") < 0) | (F.col("epss_score") > 1), F.lit("EPSS_OUT_OF_RANGE")),
    F.when((F.col("epss_percentile") < 0) | (F.col("epss_percentile") > 1), F.lit("PERCENTILE_OUT_OF_RANGE")),
)

epss_df = (
    filtered
    .withColumn("dq_flags", dq_flags)
    .withColumn("source", F.lit("epss_official"))
    .withColumn("batch_id", F.lit(batch_id))
    .withColumn("ingested_at", F.lit(INGESTED_AT))
)
epss_count = epss_df.count()


In [ ]:
if table_exists(TBL["epss"]):
    target = DeltaTable.forName(spark, TBL["epss"])
    (
        target.alias("t")
        .merge(epss_df.alias("s"), "t.cve_id = s.cve_id AND t.snapshot_date = s.snapshot_date")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    write_overwrite(epss_df, TBL["epss"], partition_by=["snapshot_date"])

log_batch_run("build_silver_epss", "SUCCESS", record_count=epss_count)
log_dq_metric("build_silver_epss", "cve_epss_match_count", epss_count, passed=True,
              detail=f"run_mode={run_mode}")
print(f"[build_silver_epss] mode={run_mode} rows={epss_count} batch_id={batch_id}")
